# Prussian FST — Corpus Coverage Analysis

Measures sentence-level binary coverage of the Twanksta-based FST against the YouTube corpus.

In [ ]:
import json
import re
import unicodedata
from collections import defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

## 1. Load data

In [ ]:
ROOT = Path.cwd().parent.parent  # project root
TWANKSTA = Path('/home/strfry/projekte/prussian-corpus/parsed/twanksta_entries.json')
CORPUS_SENTENCES = Path('/home/strfry/projekte/prussian-corpus/parsed/youtube_corpus_sentences.json')
TWANKSTA_VERBS = ROOT / 'twanksta'

SKIP_VIDEOS = {'qLwBCWtMuH8'}

In [ ]:
def fold(s):
    s = unicodedata.normalize('NFD', s)
    s = ''.join(c for c in s if not unicodedata.combining(c))
    return unicodedata.normalize('NFC', s.lower())

In [ ]:
def load_twanksta_all_forms():
    forms = set()
    lemma_forms = defaultdict(set)

    raw = json.loads(TWANKSTA.read_text(encoding='utf-8'))
    for e in raw:
        word = e.get('word', '')
        if not word or ' ' in word or '/' in word:
            continue
        forms.add(fold(word))
        lemma_forms[word].add(word)
        decl = e.get('forms', {}).get('declension', [])
        for g in decl:
            for c in g.get('cases', []):
                for num in ('singular', 'plural'):
                    f = c.get(num, '')
                    if f and ' ' not in f and '/' not in f:
                        forms.add(fold(f))
                        lemma_forms[word].add(f)
        parts = e.get('forms', {}).get('participles', [])
        for p in parts:
            f = p.get('form', '')
            if f and ' ' not in f and '/' not in f:
                forms.add(fold(f))
                lemma_forms[word].add(f)

        for mood in ('indicative', 'optative', 'imperative', 'subjunctive'):
            val = e.get('forms', {}).get(mood)
            if isinstance(val, list):
                for tense_entry in val:
                    if isinstance(tense_entry, dict):
                        for entry in tense_entry.get('forms', []):
                            f = entry.get('form', '')
                            if f and ' ' not in f and '/' not in f and '\n' not in f:
                                f = f.strip()
                                forms.add(fold(f))
                                lemma_forms[word].add(f)
                        f = tense_entry.get('form', '')
                        if f and ' ' not in f and '/' not in f and '\n' not in f:
                            f = f.strip()
                            forms.add(fold(f))
                            lemma_forms[word].add(f)
            elif isinstance(val, str):
                f = val.strip()
                if f and ' ' not in f and '/' not in f and '\n' not in f:
                    forms.add(fold(f))
                    lemma_forms[word].add(f)

    for d in sorted(TWANKSTA_VERBS.iterdir()):
        if not d.name[0].isdigit():
            continue
        vj = d / 'verb.json'
        if not vj.exists():
            continue
        vdata = json.loads(vj.read_text(encoding='utf-8'))
        lemma = vdata['lemma']
        lemma_forms[lemma].add(lemma)
        forms.add(fold(lemma))
        for tense in ('present', 'preterite'):
            for key in ('1sg', '2sg', '3sg', '1pl', '2pl', '3pl'):
                f = vdata['forms'].get(tense, {}).get(key, '')
                if f and ' ' not in f and '/' not in f and '\n' not in f:
                    f = f.strip()
                    forms.add(fold(f))
                    lemma_forms[lemma].add(f)

    return forms, lemma_forms

print('Loading Twanksta forms...')
tw_forms, lemma_forms = load_twanksta_all_forms()
print(f'  {len(tw_forms)} forms, {len(lemma_forms)} lemmas')

In [ ]:
def gemination_variants(form):
    results = set()
    for m in re.finditer(r'(.)\1', form):
        alt = form[:m.start()] + m.group(1) + form[m.end():]
        results.add(alt)
    for i in range(len(form)):
        if i+1 < len(form) and form[i] == form[i+1]:
            continue
        alt = form[:i+1] + form[i] + form[i+1:]
        if alt != form:
            results.add(alt)
    return results

def vowel_variants(form):
    vpairs = {'a':'ā','ā':'a','e':'ē','ē':'e','i':'ī','ī':'i',
              'o':'ō','ō':'o','u':'ū','ū':'u'}
    results = set()
    for i, ch in enumerate(form):
        if ch in vpairs:
            alt = form[:i] + vpairs[ch] + form[i+1:]
            results.add(alt)
    return results

# Build correctable lookup: folded forms that can be reached via gemination/vowel changes
print('Building correctable lookup...')
correctable = set()
for f in tw_forms:
    for v in gemination_variants(f):
        if v not in tw_forms:
            correctable.add(v)
    for v in vowel_variants(f):
        if v not in tw_forms:
            correctable.add(v)
print(f'  {len(correctable)} correctable variants')

## 2. Classify corpus sentences

In [ ]:
def extract_prussian_tokens(text):
    if re.match(r'^[A-Z]{2,5}:', text):
        if not text.startswith('PR:'):
            return []
        text = text[3:].lstrip()
    text = text.split('//')[0]
    text = re.sub(r'\[[^\]]*\]', '', text)
    text = re.sub(r'\([^)]*\)', '', text)
    tokens = []
    for tok in text.split():
        tok = tok.lstrip('=/')
        tok = tok.strip('.,!?;:()[]{}«»"\' \t')
        if tok:
            tokens.append(tok)
    return tokens

def classify_form(ftok):
    """Return 0=exact, 1=correctable, 2=missing"""
    if ftok in tw_forms:
        return 0
    if ftok in correctable:
        return 1
    return 2

print('Loading corpus sentences...')
raw = json.loads(CORPUS_SENTENCES.read_text(encoding='utf-8'))
print(f'  {len(raw)} entries')

results = []  # list of (has_translation, covered_exact, covered_corrected, total_tokens, tokens)
token_counts = {'exact': 0, 'correctable': 0, 'missing': 0}

for e in raw:
    # Skip unreliable videos
    sources = e.get('sources', [])
    if sources and all(s.get('video_id') in SKIP_VIDEOS for s in sources):
        continue
    
    text = e.get('text', '')
    tokens = extract_prussian_tokens(text)
    if not tokens:
        continue
    
    has_translation = bool(e.get('translations'))
    
    classifications = [classify_form(fold(t)) for t in tokens]
    covered_exact = all(c == 0 for c in classifications)
    covered_corrected = all(c <= 1 for c in classifications)
    
    for c in classifications:
        if c == 0:
            token_counts['exact'] += 1
        elif c == 1:
            token_counts['correctable'] += 1
        else:
            token_counts['missing'] += 1
    
    results.append((has_translation, covered_exact, covered_corrected, len(tokens), tokens, classifications))

print(f'  {len(results)} sentences analyzed')
print(f'  Tokens: exact={token_counts["exact"]}, correctable={token_counts["correctable"]}, missing={token_counts["missing"]}')

## 3. Coverage by sentence type

In [ ]:
# Split by translation type
with_trans = [r for r in results if r[0]]
without_trans = [r for r in results if not r[0]]

def coverage_stats(group, label):
    n = len(group)
    exact_covered = sum(1 for _, c, _, _, _, _ in group if c)
    corr_covered = sum(1 for _, _, c, _, _, _ in group if c)
    print(f'{label} ({n} Sätze):')
    print(f'  Exakt abgedeckt: {exact_covered} ({exact_covered/n*100:.1f}%)')
    print(f'  Mit Korrektur:   {corr_covered} ({corr_covered/n*100:.1f}%)')
    return exact_covered, corr_covered, n

print('== Coverage nach Satz-Typ ==\n')
e1, c1, n1 = coverage_stats(with_trans, 'Mit Übersetzung')
e2, c2, n2 = coverage_stats(without_trans, 'Ohne Übersetzung')
e3, c3, n3 = coverage_stats(results, 'Gesamt')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Bar chart: exact vs corrected by type
ax = axes[0]
labels = ['With translation', 'Without translation', 'Total']
exact_bars = [e1/n1*100, e2/n2*100, e3/n3*100]
corr_bars = [c1/n1*100, c2/n2*100, c3/n3*100]
x = np.arange(len(labels))
w = 0.35
bars1 = ax.bar(x - w/2, exact_bars, w, label='Exact coverage', color='#2e86ab')
bars2 = ax.bar(x + w/2, corr_bars, w, label='+ Gemination correction', color='#a23b72')
ax.set_ylabel('Sentences covered (%)')
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.legend()
ax.set_ylim(0, 100)
for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            f'{bar.get_height():.1f}%', ha='center', fontsize=11)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            f'{bar.get_height():.1f}%', ha='center', fontsize=11)

# Pie chart: token coverage breakdown
ax = axes[1]
sizes = [token_counts['exact'], token_counts['correctable'], token_counts['missing']]
colors = ['#2e86ab', '#a23b72', '#e8c547']
labels2 = [f'Exact ({token_counts["exact"]})',
           f'Correctable ({token_counts["correctable"]})',
           f'Missing ({token_counts["missing"]})']
ax.pie(sizes, labels=labels2, colors=colors, autopct='%1.1f%%')
ax.set_title('Token-level coverage')

plt.suptitle('YouTube Corpus — FST Coverage', fontsize=16)
plt.tight_layout()
plt.savefig('/tmp/coverage_overview.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Histogram: sentences by missing-token count

In [ ]:
# Count missing tokens per sentence
missing_counts = []
for r in results:
    n_missing = sum(1 for c in r[5] if c == 2)
    missing_counts.append(n_missing)

fig, ax = plt.subplots(figsize=(12, 5))
bins = range(0, max(missing_counts) + 2)
ax.hist(missing_counts, bins=bins, color='#2e86ab', edgecolor='white', linewidth=0.5)
ax.set_xlabel('Missing tokens per sentence')
ax.set_ylabel('Number of sentences')
ax.set_title('Sentences by number of missing tokens')
ax.set_yscale('log')
plt.tight_layout()
plt.savefig('/tmp/missing_per_sentence.png', dpi=150, bbox_inches='tight')
plt.show()

covered_exact = sum(1 for c in missing_counts if c == 0)
covered_corr = sum(1 for c in missing_counts if c == 0)  # same as above for our definition
one_missing = sum(1 for c in missing_counts if c == 1)
print(f'Fully covered: {covered_exact} ({covered_exact/len(missing_counts)*100:.1f}%)')
print(f'1 token missing: {one_missing} ({one_missing/len(missing_counts)*100:.1f}%)')
print(f'2+ tokens missing: {sum(1 for c in missing_counts if c >= 2)}')

## 5. Top missing tokens (freq)

In [ ]:
from collections import Counter

missing_token_counts = Counter()
for r in results:
    for i, c in enumerate(r[5]):
        if c == 2:
            missing_token_counts[r[4][i].lower()] += 1

print('Top 30 missing tokens (raw, no correction possible):')
print(f'{"Token":<20} {"Frequency":>10}')
print('-' * 32)
for token, freq in missing_token_counts.most_common(30):
    print(f'{token:<20} {freq:>10}')

## 6. Findings summary

Export the find_missing results as a table.

In [ ]:
import subprocess
result = subprocess.run(
    ['python3', 'scripts/find_missing.py'],
    capture_output=True, text=True, cwd=Path.cwd())
print(result.stdout)

In [ ]:
print('\nGenerated coverage report.')